# M08-02 — Facturas embebidas: partir y cruzar

[← Anterior](02-lab-json-anidado-schema.ipynb) · [Siguiente →](../M03-transformacion-datos/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

**Extra.** Un dump de facturación trae la factura **dentro** de la cuenta (modelo embebido, no dos CSV). Tras ingerir: montas **dos DataFrames** (cuentas + facturas) y los **cruzas** como en M04.

No toca `fact_lines`. Hazlo después de M08-01 (ya viste `explode`) o, como mínimo, de M02 + la teoría de este módulo.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M08-02-embedding-facturas.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


## Qué queremos conseguir (léelo antes)

En M04 las facturas **no existen**: hay `orders.csv` y `customers.csv`. El pedido trae `customer_id`. Eso es **relacionado**: dos tablas, un join.

En CMS / Mongo / “el JSON que tira facturación”, lo normal es **un documento por cuenta** y, dentro, `invoices: [ … ]`. La factura **no tiene fichero propio**. Eso es **embebido**.

Spark no hace `cuenta.invoices` como un driver de Mongo. El flujo es:

1. **Ingesta** — lees el JSONL. Un DataFrame. Columna `invoices` = array.
2. **Partir** — un DF a grano cuenta (no explotas) y otro a grano factura (`explode`).
3. **Cruzar** — `join` por `account_id`, *después*. El JSON ya no está; son tablas.

Si solo haces `show()` del documento, “no demuestra nada”: ves un array y no sabes cuántas facturas hay. La prueba son los **counts de grano** (40 cuentas ≠ 75 facturas ≠ 112 líneas).

El encabezado de los pasos dice “con tus palabras”. **Aquí no.** Copia el bloque y rellena `___`.

Fichero: `data/raw/billing_embedded.jsonl`. Si no está: `python3 scripts/generate_novashop.py`.


### Paso 1 — Ingerir el documento (aún embebido)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Copia y rellena:

```
Paso 1. Un JSON = una cuenta. invoices es un array, no una tabla.
docs = ___ filas (40). printSchema: account struct, invoices array.
vat nulo = ___ (4). Esto todavía NO es un join.
```

JSONL: **no** uses `multiLine` (eso era el array de productos).

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `documentos 40`. Schema con `account` struct e `invoices` array de structs (y `lines` dentro). `vat nulo` **4**.

**Por qué este paso.** Si el schema es todo string y no hay `array`, no es este fichero (o usaste `multiLine`).

**Si no sale.** PATH / 0 filas → `python3 scripts/generate_novashop.py`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, size, explode, explode_outer, sum as fsum
from paths import ensure_dirs

spark = get_spark("novashop-m08")
docs = spark.read.json(str(RAW / "billing_embedded.jsonl"))
print("documentos", docs.count())
docs.printSchema()
print("vat nulo", docs.where(col("account.vat").isNull()).count())
docs.select("account_id", "account.legal_name", size("invoices").alias("n_inv")).show(8)


### Paso 2 — Mirar tres cuentas **sin** explotar (tres casos)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

El array se ve distinto según la cuenta. Ejecuta y copia:

```
C0001: invoices = []          → cuenta sin facturas.
C0006: hay factura, lines []  → factura vacía por dentro (otro grano).
C0007: factura con al menos una línea.
```

Esto es el modelo embebido en crudo: **todo en el mismo JSON**.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** C0001 lista vacía. C0006 un `invoice_id` con `lines=[]`. C0007 trae `sku`/`amount`.

**Por qué este paso.** Si las tres se ven iguales, estás filtrando mal el `account_id`.


In [ ]:
print("=== C0001 (sin facturas) ===")
docs.where(col("account_id") == "C0001").select("account_id", "invoices").show(truncate=False)
print("=== C0006 (factura sin líneas) ===")
docs.where(col("account_id") == "C0006").select("account_id", "invoices").show(truncate=False)
print("=== C0007 (factura con línea) ===")
docs.where(col("account_id") == "C0007").select("account_id", "invoices").show(truncate=False)


### Paso 3 — DataFrame 1 — grano **cuenta** (no explotas)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Copia:

```
Paso 3. cuentas tiene ___ filas (40), una por JSON.
n_invoices=0 son ___ (5): C0001–C0005.
size(invoices) NO duplica filas: seguimos a grano cuenta.
```

Esto es “la tabla de clientes” que **montarías** si el origen hubiera sido relacionado.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `cuentas 40`. `sin facturas` **5**. El `show` lista `n_invoices` 0 en las primeras.

**Por qué este paso.** `size` cuenta elementos del array; no es `explode`.


In [ ]:
cuentas = docs.select(
    col("account_id"),
    col("account.legal_name").alias("legal_name"),
    col("account.vat").alias("vat"),
    col("account.billing.country").alias("country"),
    size("invoices").alias("n_invoices"),
)
print("cuentas", cuentas.count())
print("sin facturas", cuentas.where(col("n_invoices") == 0).count())
cuentas.orderBy("account_id").show(8)


### Paso 4 — DataFrame 2 — grano **factura** (`explode`)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Copia:

```
Paso 4. explode(invoices) = ___ filas (75). Distinct cuentas = ___ (35).
Se cayeron las 5 de invoices [].
Ese DF es la “tabla de facturas” que no venía en un fichero aparte.
```

`account_id` se **repite** en cada factura de la misma cuenta: esa es la clave para el join de después.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `facturas 75`. Distinct **35**. `n_lines=0` **3** (una es C0006). C0001 **no** sale.

**Por qué este paso.** 75 ≠ 40: cambió el grano. Si te da 40, no explotaste.


In [ ]:
inv_raw = docs.select("account_id", explode("invoices").alias("inv"))
facturas = inv_raw.select(
    "account_id",
    col("inv.invoice_id").alias("invoice_id"),
    col("inv.issued").alias("issued"),
    col("inv.status").alias("status"),
    size("inv.lines").alias("n_lines"),
)
print("facturas", facturas.count())
print("cuentas distintas", facturas.select("account_id").distinct().count())
print("facturas con 0 líneas", facturas.where(col("n_lines") == 0).count())
facturas.where(col("account_id").isin("C0006", "C0007", "C0011")).orderBy("account_id", "invoice_id").show(truncate=False)


## Prueba tú — explode vs explode_outer (el left del array)

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Copia:

```
explode_outer de invoices = ___ (80 = 75 facturas + 5 cuentas vacías).
Las 5 extra tienen invoice_id nulo. Eso es el left “gratis” del array.
```

**Qué tienes que ver.** **80** filas. **5** `invoice_id` nulos (C0001–C0005). C0006 **sí** tiene id (`F0006-01`) aunque `lines` esté vacío.


In [ ]:
outer_inv = docs.select("account_id", explode_outer("invoices").alias("inv"))
print("explode_outer facturas", outer_inv.count())
print(
    "invoice_id nulo",
    outer_inv.where(col("inv.invoice_id").isNull()).count(),
)
outer_inv.where(col("account_id") <= "C0006").select(
    "account_id", col("inv.invoice_id").alias("invoice_id")
).orderBy("account_id").show()


### Paso 5 — Todavía más fino: grano **línea** (array dentro del array)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

La línea de factura está embebida **otra vez** (`inv.lines`). Mismo truco.

Copia:

```
explode de lines = ___ (112). Distinct facturas = ___ (72 = 75 − 3 vacías).
C0006 desaparece de las líneas (tenía factura, cero lines).
```

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `líneas 112`. Facturas distintas **72**. C0006 → **0** filas. C0007 enseña P007 / 10.7.

**Por qué este paso.** Tres granos: 40 cuentas, 75 facturas, 112 líneas. Mezclarlos en un `show` del JSON original es lo que “no se entiende”.


In [ ]:
lineas = inv_raw.select(
    "account_id",
    col("inv.invoice_id").alias("invoice_id"),
    explode("inv.lines").alias("line"),
).select(
    "account_id",
    "invoice_id",
    col("line.sku").alias("sku"),
    col("line.qty").alias("qty"),
    col("line.amount").alias("amount"),
)
print("líneas", lineas.count())
print("facturas distintas", lineas.select("invoice_id").distinct().count())
print("¿está C0006?", lineas.where(col("account_id") == "C0006").count())
lineas.where(col("account_id") == "C0007").show()


### Paso 6 — Cruzar **después** de ingerir (como si fueran relacionados)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Ya no hay JSON. Hay `cuentas` y `facturas` con `account_id`. Eso es M04.

Copia:

```
inner  = ___ (75): solo quien tiene factura.
left desde cuentas = ___ (80): las 5 sin factura quedan con invoice_id nulo.
left ≠ inner. Si usas inner para un padrón de cuentas, pierdes C0001–C0005.
```

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `inner 75` · `left 80` · nulos de factura **5**. En el `show`, C0001–C0005 salen con `invoice_id` null; C0006 y C0007 no.

**Por qué este paso.** El join lo armas tú. El dump embebido **no** traía dos ficheros.


In [ ]:
inner_ = cuentas.join(facturas, "account_id", "inner")
left_ = cuentas.join(facturas, "account_id", "left")
print("inner", inner_.count())
print("left ", left_.count())
print("left con factura nula", left_.where(col("invoice_id").isNull()).count())
left_.where(col("account_id") <= "C0007").orderBy("account_id", "invoice_id").show()


### Paso 7 — KPI tras el cruce: GMV por cuenta (desde las **líneas**)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

El dinero está en `lineas.amount`, no en la cuenta. Agrupas líneas y haces **left** a cuentas: quien no facturó sigue saliendo.

Copia:

```
GMV nulo = ___ (6): 5 sin facturas + C0006 (factura sin líneas).
Cuentas con GMV = 34. El inner del GMV las perdería.
```

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `padrón 40`. `GMV nulo` **6** (C0001–C0006). C0007 tiene GMV **10.7**.

**Por qué este paso.** Si haces inner cuentas ⋈ gmv, C0001–C0006 desaparecen del padrón. Mismo error que el inner de M04 con huérfanos.


In [ ]:
gmv = lineas.groupBy("account_id").agg(fsum("amount").alias("gmv"))
padron = cuentas.join(gmv, "account_id", "left")
print("padrón", padron.count())
print("GMV nulo", padron.where(col("gmv").isNull()).count())
padron.where(col("gmv").isNull()).select("account_id", "n_invoices", "gmv").orderBy("account_id").show()
padron.where(col("account_id") == "C0007").show()


### Paso 8 — Persistir las tablas y volver a cruzar (ya no es JSON)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Escribes tres Parquet en staging. Al releer, el join **no** sabe que un día fueron un documento. Eso es “tratarlo **tras** la ingesta”.

Copia:

```
Releer cuentas/facturas/líneas: 40 / 75 / 112.
left otra vez = 80. El origen embebido ya no está.
```

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `40 75 112`. left **80**. C0001 releído con `invoice_id` nulo.

**Por qué este paso.** A partir de aquí el pipeline es el de siempre (join, KPI, Parquet). El JSON anidado ya hizo su trabajo.


In [ ]:
ensure_dirs()
cuentas.write.mode("overwrite").parquet(str(STAGING / "billing_accounts"))
facturas.write.mode("overwrite").parquet(str(STAGING / "billing_invoices"))
lineas.write.mode("overwrite").parquet(str(STAGING / "billing_lines"))

c2 = spark.read.parquet(str(STAGING / "billing_accounts"))
f2 = spark.read.parquet(str(STAGING / "billing_invoices"))
l2 = spark.read.parquet(str(STAGING / "billing_lines"))
print("re-cuentas", c2.count(), "re-facturas", f2.count(), "re-líneas", l2.count())
print("left tras Parquet", c2.join(f2, "account_id", "left").count())
c2.join(f2, "account_id", "left").where(col("account_id") == "C0001").show()


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Este bloque, rellenado (no un ensayo):

```
Dump embebido: 1 JSON = 1 cuenta, facturas dentro.
Tras ingesta: cuentas 40, facturas 75, líneas 112.
explode tira las 5 cuentas vacías; explode_outer las deja (80).
inner join 75; left 80. GMV nulo 6 (5 sin factura + C0006 sin líneas).
Parquet 40/75/112; el join posterior ya no ve el JSON.
```


## Mejora — Paid vs pending a grano factura

Sobre `facturas` (no sobre líneas): `groupBy("status").count()`. Markdown: ¿cuántas `paid`? No uses `docs` ni `explode` otra vez: ya partiste el documento.

Si te atasca, el código está en la celda siguiente.


In [ ]:
facturas.groupBy("status").count().orderBy("status").show()


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| 40 facturas | No explotaste `invoices` | `explode("invoices")` |
| explode = 40 | Usaste `size` o no el array | `size` no duplica filas |
| C0001 en facturas | Usaste `explode_outer` y lo llamaste explode | Inner explode las tira |
| GMV nulo 5 | Olvidaste C0006 (factura sin líneas) | El grano línea ≠ grano factura |
| inner = left | Todas las cuentas tenían array no vacío | Este dump tiene 5 `[]` |
| No está el jsonl | Generador viejo | `python3 scripts/generate_novashop.py` |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M03 — o vuelve al pipeline de pedidos](../M03-transformacion-datos/01-teoria.ipynb).
